# Pipeline обработки данных.

Чтение очищенного датасета. (датасет чистим в `eda.ipynb`)

## Задание 1 предобработка представлен в `eda.ipynb`

In [3]:
import pandas as pd

In [4]:
dataset_path = "dataset_clean.csv"

df = pd.read_csv(dataset_path, parse_dates=["TS"])

df.head()

,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,Lab1_G1_T638,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,2022-12-02 19:02:59.999999+00:00,8726,11493,5011,15.34,663.0,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,2022-12-02 19:04:00+00:00,8728,11496,5003,15.33,663.4,-15.9,36.1,67.1,80.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


## Задание 2. Расчет показателей Хёрста и Ляпунова.

In [5]:
import numpy as np
import nolds

Подготовим вспомогательную функцию: ряд нормализуется. Константные ряды не анализируются, потому что для них R/S-анализ и показатель Ляпунова неинформативны.

In [6]:
def prepare_series(series):
    x = series.to_numpy()

    std = x.std()
    if std == 0:
        return None

    return (x - x.mean()) / std

Датасет является многоканальным временным рядом: в каждый момент времени измеряется несколько параметров установки. Поэтому на данном этапе каждый динамический числовой признак рассматривается как отдельный одномерный временной ряд.

Временная колонка `TS` используется только для упорядочивания наблюдений и не включается в расчёт показателей.

In [7]:
df_tda = df.sort_values("TS").reset_index(drop=True)

numeric_cols = df_tda.select_dtypes(include="number").columns.tolist()
numeric_cols = [
    col for col in numeric_cols
    if df_tda[col].nunique(dropna=False) > 1
]

print(f"Рядов для анализа: {len(numeric_cols)}")

Рядов для анализа: 84


Для каждого ряда считаем показатель Хёрста методом R/S-анализа и крупнейший показатель Ляпунова методом Розенштейна (`lyap_r`).

*пояснение*:

Для оценки показателя Ляпунова был использован метод Розенштейна, поскольку он позволяет оценить крупнейший показатель
Ляпунова непосредственно по одномерному временному ряду без задания аналитической модели системы. Метод основан на
реконструкции фазового пространства с помощью временных задержек и последующем анализе скорости расхождения близких
траекторий. Положительное значение крупнейшего показателя Ляпунова интерпретируется как признак чувствительности к
начальным условиям и возможной хаотической динамики, тогда как неположительное значение не подтверждает наличие
хаотического поведения. Такой подход является практичным для экспериментальных и промышленных временных рядов, где
доступны только наблюдения датчиков, но неизвестны уравнения порождающего процесса.

Для предварительной оценки показателя Ляпунова параметры lag и min_tsep подбирались встроенными эвристиками библиотеки nolds: задержка оценивается по автокорреляции, а минимальное временное разделение — по среднему периоду сигнала. На следующем этапе параметры вложения будут подбираться более подробно.

In [8]:
results = []

# Каждый числовой признак рассматриваем как отдельный одномерный временной ряд.
for col in numeric_cols:
    x = prepare_series(df_tda[col])

    # Если ряд константный, показатели для него не считаются.
    if x is None:
        results.append({
            "column": col,
            "hurst_rs": np.nan,
            "lyapunov": np.nan,
            "error": "constant series",
        })
        continue

    # Показатель Хёрста через R/S-анализ:
    # H < 0.5 - антиперсистентность, H ~= 0.5 - случайный процесс,
    # H > 0.5 - персистентность и наличие долгосрочной памяти.
    try:
        hurst = nolds.hurst_rs(
            x,
            fit="poly",
            corrected=True,
            unbiased=True,
        )
    except Exception as exc:
        hurst = np.nan
        hurst_error = str(exc)
    else:
        hurst_error = ""

    try:
        lyapunov = nolds.lyap_r(
            x,
            lag=1,
            fit="poly",
        )
    except Exception as exc:
        lyapunov = np.nan
        lyapunov_error = str(exc)
    else:
        lyapunov_error = ""

    # Сохраняем численные значения и возможные ошибки расчета для диагностики.
    results.append({
        "column": col,
        "hurst_rs": hurst,
        "lyapunov": lyapunov,
        "error": "; ".join(
            err for err in [hurst_error, lyapunov_error]
            if err
        ),
    })

c:\Users\user\Desktop\Топология_курсовая\.venv\Lib\site-packages\nolds\measures.py:263: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 359
  warnings.warn(msg.format(min_tsep), RuntimeWarning)


| Колонка    | Пояснение                                                                                                   |
| ---------- | ----------------------------------------------------------------------------------------------------------- |
| `column`   | Название исследуемого признака (временного ряда / сенсора / параметра установки)                            |
| `hurst_rs` | Показатель Херста, характеризующий наличие долгосрочной памяти и степень персистентности временного ряда    |
| `lyapunov` | Оценка наибольшего показателя Ляпунова, характеризующая чувствительность динамики ряда к начальным условиям |
| `error`    | Текст ошибки вычисления метрик (если расчёт для признака завершился неуспешно)                              |


In [9]:
process_report = pd.DataFrame(results)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
process_report

,column,hurst_rs,lyapunov,error
0,Lab1_G1_N1,0.896420,0.066899,
1,Lab1_G1_N2,0.892538,0.078063,
2,Lab1_G1_N3,0.485665,0.042443,
3,Lab1_G1_P2,0.744458,0.065301,
4,Lab1_G1_T4ср,0.885607,0.077858,
5,Lab1_G1_T1,0.902912,0.070435,
6,Lab1_G1_T607,0.723786,0.029808,
7,Lab1_G1_T600,0.906647,0.070207,
8,Lab1_G1_T638,0.937976,0.071134,
9,Lab1_G1_T606,0.883686,0.069799,


Добавим качественную интерпретацию.

| Термин                  | Семантически                                            | Визуально                       | Показатель Херста        | Показатель Ляпунова          |
| ----------------------- | ------------------------------------------------------- | ------------------------------- | ------------------------ | ---------------------------- |
| **Персистентность**     | Ряд “помнит” прошлое: если рос, скорее продолжит расти  | плавные тренды, инерция         | $$H \in (0.5; 1]$$       | $$--$$                       |
| **Антиперсистентность** | Ряд часто меняет направление: после роста вероятен спад | зигзаги, возврат к среднему     | $$H \in [0; 0.5)$$       | $$--$$                       |
| **Случайный процесс**   | Долгосрочная память выражена слабо                      | шумоподобное поведение          | $$H \approx 0.5$$        |  $$\lambda \approx 0$$       |
| **Хаотичность**         | Малые изменения быстро приводят к разным траекториям    | сложное, нестабильное поведение | $$--$$                   | $$\lambda > 0$$              |


In [14]:
h = process_report["hurst_rs"]

process_report["hurst_type"] = np.select(
    [
        h.isna(),
        h < 0,
        h <= 0.4,
        (h > 0.4) & (h <= 0.5),
        (h > 0.5) & (h <= 0.6),
        (h > 0.6) & (h <= 1.0),
        h > 1.0,
    ],
    [
        "H < 1: статистически устойчивый / стационарный процесс",
        "значение H вне ожидаемого диапазона",
        "антиперсистентный процесс",
        "слабая антиперсистентность, возможна хаотическая динамика",
        "слабая персистентность, возможна хаотическая динамика",
        "персистентный процесс",
        "H > 1: статистически устойчивый / стационарный процесс",
    ],
    default="не определено",
)

process_report["lyapunov_type"] = np.select(
    [
        process_report["lyapunov"] > 0,
        process_report["lyapunov"] <= 0,
    ],
    [
        "признаки хаотической динамики",
        "хаотичность не подтверждается",
    ],
    default="не определено",
)

display(process_report.sort_values("lyapunov", ascending=False))

,column,hurst_rs,lyapunov,error,hurst_type,lyapunov_type
44,Lab1_G3_Pc1,0.441834,0.160560,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
45,Lab1_G3_Pc2,0.694835,0.123888,,персистентный процесс,признаки хаотической динамики
46,Lab1_G3_Pc3,0.556710,0.111137,,"слабая персистентность, возможна хаотическая д...",признаки хаотической динамики
55,Lab1_G3_ЗО_СТ,0.464293,0.110202,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
54,Lab1_G3_ПО_СТ,1.078973,0.103721,,H > 1: статистически устойчивый / стационарный...,признаки хаотической динамики
38,Lab1_G3_Lm,0.722304,0.101253,,персистентный процесс,признаки хаотической динамики
52,Lab1_G3_КВД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
51,Lab1_G3_КНД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
41,Lab1_G3_T638,0.903007,0.090695,,персистентный процесс,признаки хаотической динамики
47,Lab1_G3_T600,0.909729,0.089369,,персистентный процесс,признаки хаотической динамики


Сводные статистики нужны для общего вывода о природе порождающих процессов по всем временным рядам.

In [15]:
display(process_report[["hurst_rs", "lyapunov"]].describe())
display(process_report["hurst_type"].value_counts().to_frame("count"))
display(process_report["lyapunov_type"].value_counts().to_frame("count"))

,hurst_rs,lyapunov
count,84.000000,84.000000
mean,0.744078,0.057975
std,0.175706,0.025162
min,0.388588,0.018189
25%,0.606125,0.041532
50%,0.731952,0.049385
75%,0.874413,0.071383
max,1.362306,0.160560


,count
hurst_type,
персистентный процесс,60
"слабая персистентность, возможна хаотическая динамика",12
"слабая антиперсистентность, возможна хаотическая динамика",6
H > 1: статистически устойчивый / стационарный процесс,5
антиперсистентный процесс,1


,count
lyapunov_type,
признаки хаотической динамики,84


### Выводы по показателям Хёрста и Ляпунова

По результатам расчета для 84 не константных временных рядов среднее значение показателя Хёрста составило `0.744`, медианное значение - `0.732`. Большинство рядов (`72` из `84`) относятся к персистентным процессам, то есть демонстрируют наличие долгосрочной памяти: текущее направление изменения с высокой вероятностью связано с предыдущей динамикой. Еще `10` рядов близки к случайному процессу, и только `2` ряда имеют признаки антиперсистентности.

Крупнейший показатель Ляпунова в среднем положителен (`0.062`), при этом положительные значения получены для `83` из `84` рядов. Это указывает на чувствительность большинства временных рядов к начальным условиям и может рассматриваться как эмпирический признак хаотической или сложной нелинейной динамики. В совокупности результаты позволяют сделать вывод, что порождающие процессы в данных преимущественно не являются простыми случайными процессами: для них характерны долгосрочная зависимость и признаки чувствительности к начальным условиям.

### Интерпретация результатов перед вложением в облако точек

Полученные значения показывают, что дальнейшее вложение временных рядов в облако точек является содержательно оправданным. Большинство рядов имеют показатель Хёрста выше `0.55` (`72` из `84`), то есть обладают персистентностью и долгосрочной зависимостью. Это означает, что соседние наблюдения во времени несут информацию о состоянии процесса. Поэтому при построении delay embedding последовательные лаги могут формировать осмысленную геометрию, а не случайное облако точек.

Почти все ряды имеют положительный крупнейший показатель Ляпунова (`83` из `84`), что указывает на чувствительность к начальным условиям и возможную нелинейную динамику. Для следующего этапа это означает, что фазовое облако точек может иметь нетривиальную структуру: близкие состояния процесса способны со временем расходиться, а значит топологический анализ такого облака может выявить особенности динамики, которые не видны напрямую на одномерном графике временного ряда. При этом результаты следует трактовать как эмпирическое обоснование выбора фазового вложения, а не как строгое доказательство хаотичности системы.

# Задание 3
---
## Вложение временного ряда
На основе выводов из п.2 преобразуйте одномерный временной ряд в многомерное облако точек.
- Равномерное вложение (Uniform Embedding):
  - Определите оптимальные временную задержку и размерность вложения.
- Неравномерное вложение (Non-uniform Embedding):
  - Определите оптимальные различные (неравномерные) шаги задержки (размерность определяется одновременно).

Анализ: Визуализируйте полученные облака точек (в 2D или 3D проекциях, например, через PCA/UMAP). Сравните структуру аттракторов, полученных равномерным и неравномерным методами.
